# PP-OCRv4 Mobile Rec 訓練（Colab GPU）

本筆記本用於在 Google Colab 上訓練 PP-OCRv4 mobile **文字辨識 (rec)** 模型。

**前置條件：** Det 模型已訓練完成。

**Rec 標注格式：**
```
train/crop_001.jpg\t2025/03/15
train/crop_002.jpg\tEXP:2026.01
```

**流程：**
1. 安裝環境
2. 上傳 rec 資料集
3. 下載預訓練模型
4. 訓練 rec 模型
5. 匯出 + 轉換 ONNX
6. 下載模型

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝環境

In [ ]:
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install paddleocr paddle2onnx onnxruntime shapely pyclipper imgaug lmdb

In [ ]:
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.is_compiled_with_cuda()}")

## 2. 下載 PaddleOCR 原始碼

In [ ]:
import os
os.chdir("/content")

if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

os.chdir("/content/PaddleOCR")
!pwd

## 3. 上傳 Rec 資料集

Rec 資料集結構：
```
rec_dataset.zip
├── train/              ← 裁切後的文字圖片
├── val/
├── train_label.txt     ← 格式：圖片名\t辨識文字
└── val_label.txt
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATASET_PATH = "/content/drive/MyDrive/ocr_project/rec_dataset.zip"

!mkdir -p /content/dataset/rec
!unzip -o "{DRIVE_DATASET_PATH}" -d /content/dataset/rec/

In [ ]:
# 檢查資料集
!echo "=== 訓練集圖片數量 ==="
!ls /content/dataset/rec/train/ | wc -l
!echo "=== 驗證集圖片數量 ==="
!ls /content/dataset/rec/val/ | wc -l
!echo ""
!echo "=== train_label.txt 前 5 行 ==="
!head -5 /content/dataset/rec/train_label.txt

## 4. 下載預訓練模型

In [ ]:
!mkdir -p /content/pretrained_models

# 下載 PP-OCRv4 rec 訓練模型
!wget -nc -P /content/pretrained_models/ \
    https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_rec_train.tar

!cd /content/pretrained_models && tar xf ch_PP-OCRv4_rec_train.tar
!ls -la /content/pretrained_models/ch_PP-OCRv4_rec_train/

## 5. 建立訓練設定檔

In [ ]:
# 字元字典路徑
DICT_PATH = "/content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt"
print(f"字典路徑：{DICT_PATH}")
!wc -l {DICT_PATH}

In [ ]:
config_content = f"""
Global:
  model_name: PP-OCRv4_mobile_rec
  debug: false
  use_gpu: true
  epoch_num: 200
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: /content/output/rec/
  save_epoch_step: 10
  eval_batch_step: [0, 500]
  cal_metric_during_train: true
  pretrained_model: /content/pretrained_models/ch_PP-OCRv4_rec_train/best_accuracy
  checkpoints:
  save_inference_dir: /content/output/rec/inference/
  use_visualdl: true
  infer_img:
  character_dict_path: {DICT_PATH}
  max_text_length: &max_text_length 25
  infer_mode: false
  use_space_char: true
  distributed: false
  save_res_path: /content/output/rec/predicts.txt
  d2s_train_image_shape: [3, 48, 320]

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
            hidden_dims: 120
            kernel_size: [1, 3]
            use_guide: True
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /content/dataset/rec/train/
    label_file_list:
      - /content/dataset/rec/train_label.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - RecAug:
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - RecResizeImg:
        image_shape: [3, 48, 320]
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  loader:
    shuffle: true
    drop_last: true
    batch_size_per_card: 128
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /content/dataset/rec/
    label_file_list:
      - /content/dataset/rec/val_label.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - RecResizeImg:
        image_shape: [3, 48, 320]
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 128
    num_workers: 2
"""

with open("/content/rec_finetune.yml", "w") as f:
    f.write(config_content.strip())

print("設定檔已建立：/content/rec_finetune.yml")

## 6. 開始訓練

In [ ]:
os.chdir("/content/PaddleOCR")

!python tools/train.py -c /content/rec_finetune.yml

In [ ]:
# [可選] 從 checkpoint 恢復訓練
# !python tools/train.py \
#     -c /content/rec_finetune.yml \
#     -o Global.checkpoints=/content/output/rec/latest

## 7. 評估模型

In [ ]:
!python tools/eval.py \
    -c /content/rec_finetune.yml \
    -o Global.pretrained_model=/content/output/rec/best_accuracy

## 8. 匯出推論模型 + 轉 ONNX

In [ ]:
# 匯出 Paddle 推論模型
!python tools/export_model.py \
    -c /content/rec_finetune.yml \
    -o Global.pretrained_model=/content/output/rec/best_accuracy \
       Global.save_inference_dir=/content/output/rec/inference/

print("推論模型已匯出！")
!ls -la /content/output/rec/inference/

In [ ]:
# 轉 ONNX
!mkdir -p /content/output/rec/onnx

!python -m paddle2onnx.convert \
    --model_dir /content/output/rec/inference/ \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file /content/output/rec/onnx/ch_PP-OCRv4_rec_infer.onnx \
    --opset_version 11 \
    --enable_onnx_checker True

print("ONNX 轉換完成！")
!ls -la /content/output/rec/onnx/

In [ ]:
# 驗證 ONNX 模型
import onnxruntime as ort
session = ort.InferenceSession("/content/output/rec/onnx/ch_PP-OCRv4_rec_infer.onnx")
for inp in session.get_inputs():
    print(f"Input:  {inp.name} shape={inp.shape} type={inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name} shape={out.shape} type={out.type}")
print("\nONNX 模型驗證通過！")

## 9. 下載模型

In [ ]:
# 儲存到 Google Drive
!cp -r /content/output/rec/ /content/drive/MyDrive/ocr_project/output_rec/
print("模型已儲存到 Google Drive！")

In [ ]:
# 直接下載 ONNX
from google.colab import files
files.download("/content/output/rec/onnx/ch_PP-OCRv4_rec_infer.onnx")